# 네팔 지진 피해 예측 — 실험 ① 기존 9개 파생변수

원본 변수 + 처음 만든 9개 파생변수 묶음만 사용합니다.

- 모델: LightGBM (세 실험 모두 같은 설정)
- 평가: Macro F1, 학습 70% / 검증 15% / 테스트 15%
- 참고: 이전 실행에서 변수 54개, 등급 조정 후 검증 Macro F1 0.7182

`nepal_building_damage.csv`를 이 노트북과 같은 폴더에 두고 위에서부터 차례로 실행하세요.
(처음 한 번만: `%pip install lightgbm scikit-learn pandas`)

## 1. 데이터 불러오기와 분할

In [8]:
import numpy as np
import pandas as pd
import lightgbm as lgb

from pathlib import Path
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
from lightgbm import LGBMClassifier

SEED = 42
DATA_PATH = Path("./nepal_building_damage.csv")

# 데이터 불러오기
df = pd.read_csv(DATA_PATH)
print("원본 데이터 크기:", df.shape)

# 정답: 1, 2, 3등급 -> 0, 1, 2 (LightGBM은 0부터 시작하는 번호를 씀)
y = df["damage_grade"] - 1
X = df.drop(columns=["building_id", "damage_grade"])

# 학습 70% / 검증 15% / 테스트 15%
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=SEED, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=SEED, stratify=y_temp
)

# 행 번호를 0부터 다시 매기기 (뒤에서 행이 어긋나지 않게)
X_train = X_train.reset_index(drop=True)
X_val = X_val.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_val = y_val.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

print("학습:", X_train.shape, "검증:", X_val.shape, "테스트:", X_test.shape)


원본 데이터 크기: (260601, 40)
학습: (182420, 38) 검증: (39090, 38) 테스트: (39091, 38)


## 2. 기존 9개 파생변수

In [ ]:
geo_cols = ["geo_level_1_id", "geo_level_2_id", "geo_level_3_id"]


def add_9_features(data):
    """한 건물(한 행) 안의 정보만 쓰는 변수 -> 누수 걱정 없음"""
    data = data.copy()

    # 1. 나이가 995(이상값)인지 표시
    data["is_995"] = (data["age"] == 995).astype(int)

    # 2. 면적 대비 높이
    data["height_area_ratio"] = data["height_percentage"] / data["area_percentage"].clip(lower=1)

    # 3. 구조재료 개수
    material_cols = [c for c in data.columns if c.startswith("has_superstructure_")]
    data["material_count"] = data[material_cols].sum(axis=1)

    # 4. 부가용도 개수
    use_cols = [c for c in data.columns if c.startswith("has_secondary_use_")]
    data["secondary_use_count"] = data[use_cols].sum(axis=1)

    # 5. 층당 높이
    data["height_per_floor"] = data["height_percentage"] / data["count_floors_pre_eq"].clip(lower=1)

    # 6. 철근콘크리트(RC) / 흙 재료 개수
    data["rc_count"] = (data["has_superstructure_rc_non_engineered"]
                        + data["has_superstructure_rc_engineered"])
    data["mud_count"] = (data["has_superstructure_adobe_mud"]
                         + data["has_superstructure_mud_mortar_stone"]
                         + data["has_superstructure_mud_mortar_brick"])

    # 7. 구조 조합 (글자 두 개를 붙여 새 범주 만들기)
    data["foundation_roof"] = data["foundation_type"] + "_" + data["roof_type"]
    data["ground_other_floor"] = data["ground_floor_type"] + "_" + data["other_floor_type"]
    data["foundation_ground"] = data["foundation_type"] + "_" + data["ground_floor_type"]
    data["surface_condition"] = data["land_surface_condition"] + "_" + data["ground_floor_type"]

    # 8. RC / 흙 사용 여부 (0 또는 1)
    data["has_mud"] = (data["mud_count"] > 0).astype(int)
    data["has_rc"] = (data["rc_count"] > 0).astype(int)

    return data


def add_region_age(train, other):
    """9. 지역별 평균 나이 - 반드시 학습 데이터에서만 계산"""
    train, other = train.copy(), other.copy()

    normal_age = train[train["age"] != 995]   # 995는 평균에서 제외
    overall_mean = normal_age["age"].mean()   # 처음 보는 지역에 쓸 값

    for geo in geo_cols:
        age_mean = normal_age.groupby(geo)["age"].mean()
        train[geo + "_age_mean"] = train[geo].map(age_mean).fillna(overall_mean)
        other[geo + "_age_mean"] = other[geo].map(age_mean).fillna(overall_mean)

    return train, other


## 3. 범주형 변환과 변수 만들기

In [10]:
def to_category(train, other):
    """글자 변수와 지역 ID를 LightGBM 범주형으로 바꾸기
    (학습에 없던 값은 결측으로 처리)"""
    train, other = train.copy(), other.copy()
    cat_cols = train.select_dtypes(exclude=["number"]).columns.tolist()
    cat_cols = cat_cols + [c for c in geo_cols if c not in cat_cols]

    for col in cat_cols:
        train[col] = train[col].astype("category")
        known = train[col].cat.categories
        other[col] = pd.Categorical(other[col].where(other[col].isin(known)), categories=known)

    assert list(train.columns) == list(other.columns), "학습/검증 변수 구성이 다릅니다."
    return train, other


In [11]:
def make_features(train_raw, other_raw, y_train):
    train = add_9_features(train_raw)
    other = add_9_features(other_raw)
    train, other = add_region_age(train, other)
    return to_category(train, other)


train_feat, val_feat = make_features(X_train, X_val, y_train)
print("학습:", train_feat.shape, "검증:", val_feat.shape)


학습: (182420, 54) 검증: (39090, 54)


## 4. 학습과 검증

1등급·3등급 예측 확률에 곱할 배수를 검증 데이터에서 찾아 Macro F1을 높입니다.

In [12]:
MODEL_PARAMS = {
    "objective": "multiclass",
    "n_estimators": 800,
    "learning_rate": 0.04,
    "num_leaves": 31,
    "min_child_samples": 40,
    "subsample": 0.85,
    "subsample_freq": 1,
    "colsample_bytree": 0.85,
    "reg_lambda": 2.0,
    "cat_smooth": 30,
    "random_state": SEED,
    "n_jobs": -1,
    "verbosity": -1,
}


def find_class_weights(proba, y_true):
    """1등급·3등급 확률에 곱할 배수를 검증 데이터에서 찾기 (2등급은 1로 고정)"""
    best_score, best_weights = -1, [1.0, 1.0, 1.0]
    for w1 in np.arange(0.8, 1.81, 0.1):
        for w3 in np.arange(0.7, 1.51, 0.1):
            weights = [round(w1, 1), 1.0, round(w3, 1)]
            pred = (proba * weights).argmax(axis=1)
            score = f1_score(y_true, pred, average="macro")
            if score > best_score:
                best_score, best_weights = score, weights
    return best_score, best_weights


# 1) 학습
model = LGBMClassifier(**MODEL_PARAMS)
model.fit(
    train_feat, y_train,
    eval_set=[(val_feat, y_val)],
    callbacks=[lgb.early_stopping(70, verbose=False)],
)

# 2) 검증 성능
val_proba = model.predict_proba(val_feat)
basic_f1 = f1_score(y_val, val_proba.argmax(axis=1), average="macro")
tuned_f1, weights = find_class_weights(val_proba, y_val)

print("변수 수:", train_feat.shape[1])
print("최적 트리 수:", model.best_iteration_)
print("검증 Macro F1 (기본):", round(basic_f1, 4))
print("검증 Macro F1 (등급 배수 조정):", round(tuned_f1, 4))
print("등급별 배수 (1, 2, 3등급):", weights)

val_pred = (val_proba * weights).argmax(axis=1)
print(classification_report(y_val + 1, val_pred + 1, digits=4))


c:\Users\wls03\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


변수 수: 54
최적 트리 수: 799
검증 Macro F1 (기본): 0.7084
검증 Macro F1 (등급 배수 조정): 0.7182
등급별 배수 (1, 2, 3등급): [np.float64(1.7), 1.0, np.float64(1.5)]
              precision    recall  f1-score   support

           1     0.6190    0.6945    0.6546      3768
           2     0.7907    0.7692    0.7798     22239
           3     0.7163    0.7242    0.7202     13083

    accuracy                         0.7469     39090
   macro avg     0.7087    0.7293    0.7182     39090
weighted avg     0.7492    0.7469    0.7478     39090



## 5. 최종 테스트

> **주의:** 세 실험 중 무엇을 쓸지는 **검증 F1**로 먼저 정하세요. 테스트는 고른 실험 하나에서만 한 번 확인하는 것이 원칙입니다. 세 노트북 모두 테스트를 보고 고르면 테스트 점수가 낙관적이 됩니다.

In [13]:
# 학습 + 검증을 합쳐 다시 학습 (트리 수, 배수는 위에서 정한 값 그대로)
X_dev = pd.concat([X_train, X_val], ignore_index=True)
y_dev = pd.concat([y_train, y_val], ignore_index=True)

final_train, final_test = make_features(X_dev, X_test, y_dev)

final_params = MODEL_PARAMS.copy()
final_params["n_estimators"] = int(model.best_iteration_)

final_model = LGBMClassifier(**final_params)
final_model.fit(final_train, y_dev)

test_pred = (final_model.predict_proba(final_test) * weights).argmax(axis=1)

print("테스트 Accuracy:", round(accuracy_score(y_test, test_pred), 4))
print("테스트 Macro F1:", round(f1_score(y_test, test_pred, average="macro"), 4))
print(classification_report(y_test + 1, test_pred + 1, digits=4))
print("혼동행렬 (행: 실제, 열: 예측 / 1, 2, 3등급)")
print(confusion_matrix(y_test + 1, test_pred + 1))


테스트 Accuracy: 0.7432
테스트 Macro F1: 0.7134
              precision    recall  f1-score   support

           1     0.6146    0.6811    0.6461      3769
           2     0.7891    0.7635    0.7761     22239
           3     0.7096    0.7267    0.7180     13083

    accuracy                         0.7432     39091
   macro avg     0.7044    0.7237    0.7134     39091
weighted avg     0.7457    0.7432    0.7441     39091

혼동행렬 (행: 실제, 열: 예측 / 1, 2, 3등급)
[[ 2567  1115    87]
 [ 1456 16979  3804]
 [  154  3422  9507]]


## 6. 변수 중요도 (참고)

In [14]:
importance = pd.Series(final_model.feature_importances_, index=final_train.columns)
importance.sort_values(ascending=False).head(20)


geo_level_3_id             29625
geo_level_2_id             20129
height_area_ratio           2889
age                         2181
area_percentage             2158
geo_level_1_id              1663
geo_level_3_id_age_mean     1448
ground_other_floor          1244
height_per_floor            1088
foundation_roof              956
height_percentage            955
foundation_ground            904
count_families               739
position                     556
geo_level_2_id_age_mean      504
surface_condition            400
material_count               376
count_floors_pre_eq          298
geo_level_1_id_age_mean      263
roof_type                    252
dtype: int32